# Compact Europe Image Geolocation

A reproducible walkthrough for the course challenge. The notebook explains and inspects the tested package; the command-line scripts remain the source of truth for long runs.

## Prediction path

`image → compact CNN → shared embedding → country + coarse/fine cells + spherical residual → latitude/longitude`

The model represents coordinates as 3D unit vectors. This avoids treating +179° and −179° longitude as far apart.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from geoguesser.cells import CountryKMeansPartition
from geoguesser.config import load_config
from geoguesser.data import add_country_ids, audit_dataset, read_labels
from geoguesser.geo import metric_summary
from geoguesser.leaderboard import leaderboard_fold_assignments
from geoguesser.model import build_model, model_config_from_runtime, model_parameter_count


In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'configs').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_ROOT = PROJECT_ROOT.parents[1] / 'geo_dataset'
CONFIG_PATH = PROJECT_ROOT / 'configs' / 'efficientnet_b0_scratch.yaml'
RUN_TRAINING = False  # Long runs belong in the terminal.
config = load_config(CONFIG_PATH)
config['data']['root'] = str(DATA_ROOT)
print({'project': str(PROJECT_ROOT), 'data': str(DATA_ROOT), 'torch': torch.__version__})

## 1. Fail fast on bad data

This verifies every labelled filename, the 2,400-image holdout contract, coordinate ranges, duplicates, and the exact country set.

In [ ]:
audit = audit_dataset(DATA_ROOT)
display(pd.Series(audit))

In [ ]:
labels, country_to_id = add_country_ids(read_labels(DATA_ROOT / 'train_labels.csv'))
summary = (labels.groupby(['country', 'iso'], as_index=False)
           .agg(n=('filename', 'size'), lat_min=('lat', 'min'), lat_max=('lat', 'max'),
                lng_min=('lng', 'min'), lng_max=('lng', 'max')))
display(summary.sort_values('country'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
summary.sort_values('country').plot.bar(x='country', y='n', legend=False, ax=axes[0])
axes[0].set_title('Images per country')
for country, part in labels.groupby('country'):
    axes[1].scatter(part['lng'], part['lat'], s=5, alpha=0.5, label=country)
axes[1].set(title='Coordinate coverage', xlabel='longitude', ylabel='latitude')
axes[1].legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=7)
plt.tight_layout()

## 2. Validation is part of the model

Use the downloaded seed-42 `leaderboard/folds.csv` so the pooled out-of-fold result is comparable with the unofficial board. Run `configs/spatial_validation.yaml` separately to test whether nearby routes leak across random folds.

In [ ]:
folds = leaderboard_fold_assignments(labels, PROJECT_ROOT / 'leaderboard' / 'folds.csv')
fold_table = pd.crosstab(labels['country'], folds)
display(fold_table)
assert (fold_table > 0).all().all()

## 3. Fit geographic cells on training rows only

Country-conditioned K-means on unit-sphere coordinates creates approximately balanced cells. Medoids—not arithmetic latitude/longitude means—are the default output centers.

In [ ]:
train_df = labels.loc[folds != 0].reset_index(drop=True)
val_df = labels.loc[folds == 0].reset_index(drop=True)
fine = CountryKMeansPartition(config['geocells']['fine_target_size'], random_state=2026)
coarse = CountryKMeansPartition(config['geocells']['coarse_target_size'], random_state=2026)
train_df['fine_cell'] = fine.fit_transform(train_df)
train_df['coarse_cell'] = coarse.fit_transform(train_df)
val_df['fine_cell'] = fine.transform(val_df)
val_df['coarse_cell'] = coarse.transform(val_df)
print({'fine_cells': fine.n_cells_, 'coarse_cells': coarse.n_cells_})

In [ ]:
oracle_prediction = fine.output_centers_latlng_[val_df['fine_cell'].to_numpy()]
oracle_metrics = metric_summary(val_df[['lat', 'lng']].to_numpy(), oracle_prediction)
display(pd.Series(oracle_metrics, name='fine-cell quantization oracle'))
print('This is a resolution ceiling, not an image-model result: it uses the true country/cell assignment.')

## 4. Construct the exact submitted architecture

Torchvision reports 5.29M parameters for stock EfficientNet-B0, but most belong to its discarded ImageNet classifier. The custom model uses the feature extractor plus compact geographic heads.

In [ ]:
runtime_model_config = model_config_from_runtime(
    config, len(country_to_id), fine.n_cells_, coarse.n_cells_)
model = build_model(runtime_model_config)
parameter_count = model_parameter_count(model)
print(f'Fold model parameters: {parameter_count:,}')
assert parameter_count <= config['rules']['max_parameters']

In [ ]:
with torch.inference_mode():
    output = model(torch.randn(2, 3, 224, 224))
{name: tuple(value.shape) for name, value in output.items()}

## 5. Train one fold

The script records the rows, history, best checkpoint, out-of-fold coordinates, hardware, runtime, parameter count, and compliance metadata.

In [ ]:
command = [sys.executable, 'scripts/train.py', '--config', str(CONFIG_PATH),
           '--data-root', str(DATA_ROOT), '--fold', '0']
print(' '.join(command))
if RUN_TRAINING:
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)

## 6. Experiments, in order

1. MobileNet scratch on fold 0.
2. EfficientNet scratch on the same fold.
3. Cell sizes 20/24/32.
4. No TTA versus 180° TTA.
5. Train-only SimCLR initialization.
6. All five shared leaderboard folds for finalists.
7. Spatial stress test.

Append results to `experiments.csv`; never tune on `holdout_public/`.

In [ ]:
benchmark_command = [sys.executable, 'scripts/benchmark.py', '--configs',
    'configs/mobilenet_v3_small_scratch.yaml',
    'configs/efficientnet_b0_scratch.yaml',
    '--data-root', str(DATA_ROOT), '--fold', '0']
print(' '.join(benchmark_command))

## 7. Final fit and submission

After cross-validation fixes the architecture, epochs, and decoder settings, train on all labels with `--fold full`. Use one rule-compliant checkpoint unless the instructors explicitly clarify how the parameter cap applies to ensembles.

In [ ]:
final_commands = [
    f"{sys.executable} scripts/train.py --config {CONFIG_PATH} --data-root {DATA_ROOT} --fold full --run-dir outputs/final",
    f"{sys.executable} scripts/predict.py --checkpoint outputs/final/final_model.pt --data-root {DATA_ROOT} --output outputs/final/predictions.csv --tta rot180",
    f"{sys.executable} scripts/validate_submission.py --predictions outputs/final/predictions.csv --holdout-dir {DATA_ROOT / 'holdout_public'}",
]
print('\n'.join(final_commands))

## References

- Weyand et al., *PlaNet* (ECCV 2016).
- Haas et al., *PIGEON* (CVPR 2024).
- Chen et al., *SimCLR* (ICML 2020).
- Tan and Le, *EfficientNet* (ICML 2019).